# 02b — Phase-A full-batch gate (Day 3)

First GPU task of day 3 (§10.1): measure the REAL phase-A step —
forward+backward at full batch (256 windows → **512 SupCon views**)
with the real P×K sampler, real §3 augmentation and the real AdamW+AMP
step — replacing the day-2 gate's declared 2×-per-step approximation
(`reports/gate_day2.json`, projected ~7.01 h with ~1 h of margin).

**Pre-committed rule (§10.1):** projected phase A **> 8 h** →
escalation §5.2 BEFORE any phase-A run. Peak GPU memory is also
recorded (§5.2: activation checkpointing assumed unnecessary — this is
the day-3 verification).

Thin runner: all logic lives in `sharp_har/bench.py`.

## 1. Environment setup

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"
REPO_DIR = Path("/content/sharp-har")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

import torch

sys.path.insert(0, str(REPO_DIR))

from sharp_har.utils import set_seed, get_git_hash, read_yaml, write_json

set_seed(42)
assert torch.cuda.is_available(), "the phase-A gate is meaningless without the GPU — Runtime > Change runtime type > T4"
print("git hash:", get_git_hash(REPO_DIR))
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Mount Drive + staging (same procedure as notebook 02, §8.5)

In [ ]:
import zipfile
from google.colab import drive

drive.mount("/content/drive")

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])

if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)
print("staged:", stage_dir)

## 3. Measured phase-A step + go/no-go

Uses the unmodified `c3_supcon.yaml` (bench deep-copies it). 5 warmup
steps (cache/cuDNN autotune), 20 measured steps. The result JSON also
carries the §4.2 mandatory sampler-composition log (distinct AR-sets
and unique traces per class, trace reuses with minimum offset).

In [ ]:
from sharp_har.bench import phase_a_step_bench

cfg = read_yaml(REPO_DIR / "configs" / "c3_supcon.yaml")
gate = phase_a_step_bench(cfg, stage_dir=stage_dir, repo_dir=REPO_DIR)

print(f"s/step (512 views): {gate['s_per_step']:.3f}")
print(f"peak GPU memory: {gate['peak_mem_allocated_gb']} GiB allocated / {gate['peak_mem_reserved_gb']} GiB reserved")
print(f"projected phase A: {gate['projected_phase_a_hours']} h (rule: <= {gate['rule_phase_a_max_hours']} h)")
print("GATE:", "GO" if gate["phase_a_pass"] else "NO-GO -> escalation §5.2 BEFORE any phase-A run")

write_json(REPO_DIR / "reports" / "gate_day3_phase_a.json", gate)
print("written:", REPO_DIR / "reports" / "gate_day3_phase_a.json")

## 4. Closing the gate

1. Commit `reports/gate_day3_phase_a.json` and update `STATUS.md` in
   the same commit.
2. If NO-GO: escalation §5.2 in order — (a) 400→300 steps/epoch,
   (b) `escalation_b=True`, (c) shorter horizon with the pre-committed
   checkpoint-grid rule ⌈2H/3⌉/⌈5H/6⌉/H — recorded in the split file
   changelog, then re-run THIS notebook.
3. Recalibrate the phase-A rows of the §8.4 budget table.